# 02 – Feature exploration

Explores the candidate feature columns from the **Dec 2018 snapshot**, for the trials in `analysis.cohort`. The goal is to decide which columns are usable as features (see `docs/feature_list.md`).

## Section 1 – Load the one-per-trial columns

The cohort joined to `studies`, `designs` and `eligibilities` (each has one row per trial).

In [1]:
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  

from src.db import engine

query = text("""
    SELECT 
        -- Cohort keys and split
        c.nct_id,
        c.label,
        c.split,
        c.status_2018,
        
        -- Studies candidates
        s.phase,
        s.enrollment,
        s.enrollment_type,
        s.start_date,
        s.primary_completion_date,
        s.number_of_arms,
        s.has_dmc,
        
        -- Designs candidates
        d.allocation,
        d.masking,
        d.primary_purpose,
        d.intervention_model,
        
        -- Eligibilities candidates
        e.gender,
        e.minimum_age,
        e.maximum_age,
        e.healthy_volunteers

    FROM analysis.cohort c
    LEFT JOIN snap_2018_12.studies s ON c.nct_id = s.nct_id
    LEFT JOIN snap_2018_12.designs d ON c.nct_id = d.nct_id
    LEFT JOIN snap_2018_12.eligibilities e ON c.nct_id = e.nct_id;
""")

with engine.connect() as conn:
    df_trials = pd.read_sql(query, conn)

### Check: row count (expect 47,210, so the joins didn't multiply rows)

In [2]:
print(df_trials.shape)

(47210, 19)


## Section 2 – Explore the one-per-trial columns

### Missing values (% of rows)

In [3]:
display(((df_trials.isna().mean())*100).round(2))


nct_id                      0.00
label                      32.42
split                       0.00
status_2018                 0.00
phase                       0.00
enrollment                  0.04
enrollment_type             0.05
start_date                  0.03
primary_completion_date     0.01
number_of_arms              0.71
has_dmc                    11.87
allocation                 27.40
masking                     0.17
primary_purpose             0.31
intervention_model          0.09
gender                      0.00
minimum_age                 0.00
maximum_age                 0.00
healthy_volunteers          0.00
dtype: float64

### Categorical columns: value counts (including missing)

In [4]:
cat_cols = [
    'phase', 
    'enrollment_type', 
    'allocation', 
    'masking', 
    'primary_purpose', 
    'intervention_model', 
    'gender', 
    'healthy_volunteers'
]
for col in cat_cols:
    display(df_trials[col].value_counts(dropna=False))

phase
N/A                24060
Phase 2             7267
Phase 3             3995
Phase 1             3912
Phase 4             3787
Phase 1/Phase 2     2401
Phase 2/Phase 3      954
Early Phase 1        834
Name: count, dtype: int64

enrollment_type
Anticipated    47187
NaN               23
Name: count, dtype: int64

allocation
Randomized        29443
NaN               12936
Non-Randomized     4831
Name: count, dtype: int64

masking
None (Open Label)    28369
Single                6548
Double                5007
Quadruple             4066
Triple                3140
NaN                     80
Name: count, dtype: int64

primary_purpose
Treatment                   30787
Prevention                   4566
Other                        2820
Diagnostic                   2697
Supportive Care              2311
Basic Science                2144
Health Services Research     1109
Screening                     425
Device Feasibility            203
NaN                           148
Name: count, dtype: int64

intervention_model
Parallel Assignment        28717
Single Group Assignment    14021
Crossover Assignment        2665
Sequential Assignment       1110
Factorial Assignment         654
NaN                           43
Name: count, dtype: int64

gender
All       40617
Female     4836
Male       1757
Name: count, dtype: int64

healthy_volunteers
No                            37307
Accepts Healthy Volunteers     9888
                                 15
Name: count, dtype: int64

### Numeric columns: summary statistics

In [5]:
display(df_trials[['enrollment','number_of_arms']].describe())


,enrollment,number_of_arms
count,4.719000e+04,46877.000000
mean,4.698630e+03,2.051710
std,6.511029e+05,1.176256
min,1.000000e+00,1.000000
25%,4.000000e+01,1.000000
50%,8.000000e+01,2.000000
75%,1.950000e+02,2.000000
max,1.000000e+08,31.000000


### Age columns: what the text looks like

In [6]:
display(df_trials[['minimum_age','maximum_age']].head(10))

,minimum_age,maximum_age
0,18 Years,N/A
1,35 Years,42 Years
2,18 Years,N/A
3,18 Years,75 Years
4,35 Years,80 Years
5,18 Years,65 Years
6,18 Years,N/A
7,18 Years,70 Years
8,20 Years,80 Years
9,18 Years,N/A


### Is the missing `allocation` explained by single-group trials?

In [7]:
pd.crosstab(df_trials['allocation'].fillna('no allocation'),df_trials['intervention_model']).T

allocation,Non-Randomized,Randomized,no allocation
intervention_model,,,
Crossover Assignment,175,2483,7
Factorial Assignment,60,591,3
Parallel Assignment,3227,25451,39
Sequential Assignment,509,375,226
Single Group Assignment,850,538,12633


### The largest `enrollment` values (outlier check)

In [8]:
df_trials.sort_values(by='enrollment',ascending=False).head(10)

,nct_id,label,split,status_2018,phase,enrollment,enrollment_type,start_date,primary_completion_date,number_of_arms,has_dmc,allocation,masking,primary_purpose,intervention_model,gender,minimum_age,maximum_age,healthy_volunteers
42292,NCT01912313,NaN,train,RECRUITING,N/A,99999999.0,Anticipated,2013-08-31,2019-08-31,1.0,False,NaN,None (Open Label),Basic Science,Single Group Assignment,All,18 Years,99 Years,Accepts Healthy Volunteers
38741,NCT02421705,NaN,train,RECRUITING,N/A,99999999.0,Anticipated,2010-02-28,2099-01-31,1.0,False,NaN,None (Open Label),Basic Science,Single Group Assignment,All,18 Years,65 Years,Accepts Healthy Volunteers
24862,NCT03090321,NaN,train,RECRUITING,N/A,2000000.0,Anticipated,2015-03-31,2017-12-31,5.0,False,Randomized,Single,Prevention,Parallel Assignment,All,18 Years,N/A,Accepts Healthy Volunteers
11171,NCT03455478,NaN,test,RECRUITING,N/A,1000000.0,Anticipated,2016-01-01,2018-11-30,2.0,None,Non-Randomized,None (Open Label),Prevention,Parallel Assignment,All,60 Years,100 Years,Accepts Healthy Volunteers
2166,NCT03676764,0.0,test,NOT_YET_RECRUITING,Phase 4,447780.0,Anticipated,2019-01-31,2022-01-31,4.0,True,Randomized,Quadruple,Prevention,Factorial Assignment,All,1 Month,59 Months,Accepts Healthy Volunteers
42533,NCT01864603,0.0,train,RECRUITING,N/A,340000.0,Anticipated,2013-04-30,2020-07-31,2.0,True,Randomized,None (Open Label),Other,Parallel Assignment,All,N/A,N/A,Accepts Healthy Volunteers
16144,NCT03335072,NaN,valid,NOT_YET_RECRUITING,Phase 4,320000.0,Anticipated,2019-06-01,2022-07-01,4.0,True,Randomized,None (Open Label),Treatment,Parallel Assignment,All,N/A,N/A,No
27421,NCT03014167,0.0,train,RECRUITING,Phase 4,240000.0,Anticipated,2017-10-04,2022-12-31,2.0,True,Randomized,None (Open Label),Treatment,Single Group Assignment,All,12 Months,N/A,Accepts Healthy Volunteers
38841,NCT02392832,0.0,train,RECRUITING,N/A,240000.0,Anticipated,2015-04-30,2018-09-30,2.0,False,Randomized,Single,Other,Crossover Assignment,All,7 Months,N/A,No
16220,NCT03338244,0.0,valid,RECRUITING,Phase 4,227000.0,Anticipated,2017-10-20,2019-07-31,2.0,True,Randomized,None (Open Label),Treatment,Parallel Assignment,All,1 Month,60 Months,No


### Units used in `maximum_age`

In [9]:
df_trials['maximum_age'].str.extract(r'^\d+\s*([a-zA-Z]+)')[0].value_counts(dropna=False)

0
Years      24212
NaN        22098
Months       410
Weeks        182
Days         173
Hours         72
Year          22
Minutes       16
Month         11
Day           10
Hour           2
Week           1
Minute         1
Name: count, dtype: int64

### Units used in `minimum_age`

In [10]:
df_trials['minimum_age'].str.extract(r'^\d+\s*([a-zA-Z]+)')[0].value_counts(dropna=False)

0
Years     43510
NaN        2468
Months      638
Year        336
Weeks       129
Month       107
Days         22
Name: count, dtype: int64

### Findings – Section 2

- **Row count is 47,210** after joining `studies`, `designs` and `eligibilities`, so these joins don't multiply rows.
- **`enrollment_type` is "Anticipated" for every trial** (47,187; 23 missing). This confirms that the 2018 enrolment is the *target* size, not the actual count, so it's safe from leakage. The column itself never varies, so it's **dropped** as a feature.
- **`allocation` is 27.4% missing, and that's not random.** 12,633 of the 12,936 missing values (97.7%) are single-group trials, which have nobody to randomise. Fill with a category such as "single group".
- **`phase` = "N/A" for 51%.** These are non-drug trials (devices, behavioural…), so "N/A" is a real category, not missing data.
- **`enrollment` has extreme values.** The median is 80, but the max is 99,999,999 (a placeholder, 2 trials), and there are several values in the 200,000–2,000,000 range. Treat the placeholder as missing, and use a log scale.
- **Placeholder dates exist.** For example, `primary_completion_date` = 2099-01-31. Check for these before computing planned duration.
- **The ages are text** ("18 Years"), with units from Minutes to Years and both singular and plural forms. "N/A" means *no limit*: 46.8% have no maximum age and 5.2% have no minimum. These need parsing to a number in years, plus a "no limit" flag.
- **`healthy_volunteers` hides 15 missing values as empty strings** (`""`), which `isna()` doesn't count.
- **`has_dmc` is 11.9% missing.** Keep "missing" as its own category.